# Day 4: Probability Calibration, Threshold Optimization & SHAP Explainability

**Project**: Explainable Credit Card Fraud Detection and Risk Scoring  
**Stage**: Day 4 — Leakage-safe calibration, threshold selection, SHAP, risk scoring  
**Dataset**: Kaggle ULB Credit Card Fraud Detection (284,807 transactions)  

## Overview
1. Reload chronological splits
2. Carve calibration subset from training partition (75/25)
3. Retrain RF & XGBoost on fit subset (Day-3 frozen hyperparameters)
4. Fit sigmoid (Platt) calibrators on calibration subset
5. Compare raw vs calibrated Brier score and log-loss
6. Threshold sweep → optimal F1 threshold
7. SHAP global summary + feature importance
8. SHAP local waterfall plots (fraud & legitimate)
9. Risk scoring with `predict_risk()`


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.data_loader import load_raw_data
from src.features.data_split import create_chronological_splits
from src.models.calibration import (
    create_calibration_split, retrain_rf_on_fit, retrain_xgb_on_fit,
    fit_sigmoid_calibrator, build_calibrated_pipeline, compare_calibration_metrics,
    plot_reliability_diagrams, save_calibrated_pipeline
)
from src.models.threshold_optimizer import (
    compute_threshold_metrics, find_optimal_threshold,
    compare_f1_and_cost_thresholds, precision_recall_at_k,
    plot_threshold_curves, DEFAULT_THRESHOLDS, FP_COST, FN_COST
)
from src.explainability.shap_explainer import (
    compute_shap_values, plot_shap_summary, plot_shap_feature_importance,
    plot_shap_waterfall_local, get_expected_value
)
from src.models.risk_scorer import predict_risk

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
print('Imports OK')


## Step 1: Load Data and Recreate Splits


In [ ]:
df = load_raw_data(DATA_PATH)
train_df, val_df, test_df = create_chronological_splits(df)

TARGET = 'Class'
FEATURE_COLS = ['Time', 'Amount'] + [f'V{i}' for i in range(1, 29)]

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
X_val, y_val     = val_df[FEATURE_COLS],   val_df[TARGET]
X_test           = test_df[FEATURE_COLS]   # HELD OUT

print(f'Train: {len(train_df):,} rows | fraud={y_train.sum()}')
print(f'Val  : {len(val_df):,} rows | fraud={y_val.sum()}')
print(f'Test : {len(test_df):,} rows [STRICTLY HELD OUT]')


## Step 2–5: Leakage-Safe Calibration

**Approach**:
- Split training partition: first 75% → model fitting, last 25% → calibrator fitting.
- Re-train RF and XGBoost with **frozen Day-3 hyperparameters** on the fit subset.
- Fit `CalibratedClassifierCV(method='sigmoid')` with `FrozenEstimator` on calibration subset.
- Guarantees zero re-fitting of base estimator trees on calibration data.
- Validation data is **never** used for fitting or calibration.


In [ ]:
fit_df, calib_df = create_calibration_split(train_df, fit_ratio=0.75)
X_fit, y_fit = fit_df[FEATURE_COLS], fit_df[TARGET]
X_calib, y_calib = calib_df[FEATURE_COLS], calib_df[TARGET]

# Retrain on fit subset
rf_prep, rf_clf_raw = retrain_rf_on_fit(X_fit, y_fit, n_estimators=100, max_depth=10)
xgb_prep, xgb_clf_raw = retrain_xgb_on_fit(X_fit, y_fit)

# Fit calibrators
rf_cal_clf  = fit_sigmoid_calibrator(rf_clf_raw,  rf_prep.transform(X_calib),  y_calib)
xgb_cal_clf = fit_sigmoid_calibrator(xgb_clf_raw, xgb_prep.transform(X_calib), y_calib)

cal_rf_pipeline  = build_calibrated_pipeline(rf_prep,  rf_cal_clf)
cal_xgb_pipeline = build_calibrated_pipeline(xgb_prep, xgb_cal_clf)

rf_metrics  = compare_calibration_metrics(rf_prep,  rf_clf_raw,  cal_rf_pipeline,  X_val, y_val, 'RF')
xgb_metrics = compare_calibration_metrics(xgb_prep, xgb_clf_raw, cal_xgb_pipeline, X_val, y_val, 'XGB')

print('\nCalibration diagnostics (validation set):')
for name, m in [('RF', rf_metrics), ('XGBoost', xgb_metrics)]:
    print(f'  {name}: Brier raw={m["brier_raw"]:.5f} → cal={m["brier_cal"]:.5f} | '
          f'LogLoss raw={m["logloss_raw"]:.5f} → cal={m["logloss_cal"]:.5f}')


In [ ]:
# Load saved figure
from IPython.display import Image
Image(str(FIGURES_DIR / '13_reliability_diagram.png'), width=700)


## Step 6: Threshold Optimization (Validation Set)

**Cost assumptions (illustrative only — not real business data)**:
- FP cost = \$10 (analyst review + customer friction)
- FN cost = \$500 (assumed average fraud loss)


In [ ]:
thresholds = DEFAULT_THRESHOLDS
rf_thr_df  = compute_threshold_metrics(cal_rf_pipeline,  X_val, y_val, thresholds)
xgb_thr_df = compute_threshold_metrics(cal_xgb_pipeline, X_val, y_val, thresholds)

(rf_f1_thr, rf_f1_row), (rf_cost_thr, rf_cost_row) = compare_f1_and_cost_thresholds(rf_thr_df)
(xgb_f1_thr, xgb_f1_row), (xgb_cost_thr, xgb_cost_row) = compare_f1_and_cost_thresholds(xgb_thr_df)

print('Calibrated RF:')
print(f'  F1-optimal   : threshold={rf_f1_thr:.3f} | F1={rf_f1_row["f1"]:.4f} P={rf_f1_row["precision"]:.4f} R={rf_f1_row["recall"]:.4f} Cost=\${rf_f1_row["financial_cost"]:.0f}')
print(f'  Cost-optimal : threshold={rf_cost_thr:.3f} | Cost=\${rf_cost_row["financial_cost"]:.0f} F1={rf_cost_row["f1"]:.4f} P={rf_cost_row["precision"]:.4f} R={rf_cost_row["recall"]:.4f}')
print('\nCalibrated XGBoost:')
print(f'  F1-optimal   : threshold={xgb_f1_thr:.3f} | F1={xgb_f1_row["f1"]:.4f} P={xgb_f1_row["precision"]:.4f} R={xgb_f1_row["recall"]:.4f} Cost=\${xgb_f1_row["financial_cost"]:.0f}')
print(f'  Cost-optimal : threshold={xgb_cost_thr:.3f} | Cost=\${xgb_cost_row["financial_cost"]:.0f} F1={xgb_cost_row["f1"]:.4f} P={xgb_cost_row["precision"]:.4f} R={xgb_cost_row["recall"]:.4f}')


In [ ]:
Image(str(FIGURES_DIR / '14_threshold_curves.png'), width=900)


## Step 7–8: SHAP Explainability

> **Important**: SHAP values explain the **raw tree model's decision margin (log-odds output)**,  
> NOT the sigmoid-calibrated probabilities. Platt calibration applies a monotonic sigmoid mapping,  
> but because sigmoid scaling is non-linear and saturates in the tails, feature additivity  
> holds strictly in margin space and does not equal additive probability percentages.


In [ ]:
# Load saved SHAP figures
print('SHAP Summary Plot:')
display(Image(str(FIGURES_DIR / '15_shap_summary.png'), width=800))
print('SHAP Feature Importance:')
display(Image(str(FIGURES_DIR / '16_shap_feature_importance.png'), width=700))


In [ ]:
print('Waterfall — Fraud transaction:')
display(Image(str(FIGURES_DIR / '17_shap_waterfall_fraud.png'), width=750))
print('Waterfall — Legitimate transaction:')
display(Image(str(FIGURES_DIR / '18_shap_waterfall_legit.png'), width=750))


## Step 9: Risk Scoring with `predict_risk()`

Risk tiers:
- **Low**: fraud_probability < 0.10
- **Medium**: 0.10 ≤ fraud_probability < 0.50
- **High**: fraud_probability ≥ 0.50


In [ ]:
# Load saved calibrated pipeline and threshold
from src.models.calibration import load_calibrated_pipeline
from src.models.threshold_optimizer import load_threshold_metadata

metadata = load_threshold_metadata(ARTIFACTS_DIR / 'threshold_metadata.json')
winning_model = metadata['model_name']
opt_thr = metadata['optimal_threshold']

if 'XGBoost' in winning_model:
    pipeline = load_calibrated_pipeline(ARTIFACTS_DIR / 'calibrated_xgb_pipeline.joblib')
else:
    pipeline = load_calibrated_pipeline(ARTIFACTS_DIR / 'calibrated_rf_pipeline.joblib')

print(f'Winning model: {winning_model}')
print(f'Optimal threshold: {opt_thr:.3f} (objective: F1-score, validation set only)')

# Score 10 validation rows
sample = X_val.iloc[:10].copy()
risk_df = predict_risk(sample, pipeline, threshold=opt_thr)
display(risk_df)


## Summary

| Item | Value |
|------|-------|
| Calibration method | Platt sigmoid (`CalibratedClassifierCV`, `cv='prefit'`) |
| Calibration subset | Last 25% of training partition (chronological, disjoint) |
| Test partition | **NEVER TOUCHED** |
| Threshold selection | F1-optimal on validation set only |
| SHAP explainer | `shap.TreeExplainer` on raw classifier (not calibrated wrapper) |
| SHAP scope | Explains raw log-odds, NOT calibrated probabilities |

**Next**: Day 5 — Streamlit interactive dashboard.
